# Fase 06 — Evaluación del sistema de visión

Este cuaderno carga `salidas/evaluacion.csv` (generado por `python -m placas.evaluacion`) y guía el análisis de los resultados con preguntas, no respuestas. Corra primero:

```bash
python -m placas.generador --n 200 --semilla 42
python -m placas.evaluacion
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../salidas/evaluacion.csv")
df.head()

## 1. Localización

**Pregunta:** ¿qué porcentaje de placas se localizaron correctamente? ¿Coincide, aproximadamente, con el porcentaje que reportó `python -m placas.localizacion` en la Fase 04? Si no coincide, ¿qué podría explicar la diferencia (recuerde que el pipeline usa color y, si falla, bordes como respaldo)?

In [ ]:
porcentaje_localizadas = df["localizada"].mean() * 100
print(f"Localizadas: {porcentaje_localizadas:.1f} %")

## 2. Exactitud de OCR: por placa completa vs. por carácter

**Pregunta:** ¿por qué la exactitud por carácter suele ser más alta que la exactitud por placa completa? ¿Qué significa, en términos prácticos, que una placa esté "casi bien" leída (5 de 6 caracteres correctos) para el sistema de control de acceso del ENUNCIADO?

In [ ]:
exactitud_completa = (df["placa_esperada"] == df["placa_obtenida"]).mean() * 100
exactitud_caracter = df["exactitud_caracter"].mean() * 100
print(f"Exactitud por placa completa: {exactitud_completa:.1f} %")
print(f"Exactitud por caracter:       {exactitud_caracter:.1f} %")

## 3. Tiempo por etapa

**Pregunta:** ¿cuál etapa del pipeline (preprocesamiento, localización u OCR) consume más tiempo en promedio? ¿Ese resultado le sorprende? ¿Qué implicaría para correr el sistema en tiempo real sobre una talanquera?

In [ ]:
tiempos = df[["tiempo_preproceso_s", "tiempo_localizacion_s", "tiempo_ocr_s"]].mean()
tiempos.plot(kind="bar", title="Tiempo medio por etapa (s)")
plt.ylabel("segundos")
plt.show()
tiempos

## 4. Exactitud agrupada por degradación

**Pregunta:** ¿la exactitud por carácter baja a medida que aumenta el kernel de desenfoque (`desenfoque_kernel`)? Si agregó una nueva degradación en el reto de la Fase 01 (lluvia, sombra, placa sucia), ¿cómo agruparía por esa nueva columna en vez de por desenfoque?

In [ ]:
exactitud_por_desenfoque = df.groupby("desenfoque_kernel")["exactitud_caracter"].mean()
exactitud_por_desenfoque.plot(kind="bar", title="Exactitud por caracter segun kernel de desenfoque")
plt.ylabel("exactitud promedio")
plt.show()
exactitud_por_desenfoque

## 5. Matriz de confusión de caracteres

**Pregunta:** mirando `salidas/confusiones.png`, ¿cuáles son las tres confusiones más frecuentes? ¿Coinciden con las correcciones ya definidas en `config/reglas.yaml` (por ejemplo O↔0, B↔8)? Si encuentra una confusión frecuente que no está en la lista de correcciones, ¿la agregaría? ¿Por qué sí o por qué no (piense en falsos positivos: corregir de más también puede introducir errores)?

In [ ]:
from IPython.display import Image

Image("../salidas/confusiones.png")

## Reto de esta fase

Identifique las tres confusiones de caracteres más frecuentes (a partir de la matriz de confusión) y proponga una corrección concreta: puede ser una regla nueva en `correcciones` de `config/reglas.yaml`, un ajuste del preprocesamiento (Fase 03) o un ajuste de la localización (Fase 04). Justifique la propuesta con los datos de este cuaderno, no solo con intuición.